In [1]:
import numpy as np


class XGBoostNode:

  def __init__(
      self,
      feature=None,
      threshold=None,
      left=None,
      right=None,
      *,
      value=None,
  ):
    self.feature = feature
    self.threshold = threshold
    self.left = left
    self.right = right
    self.value = value

  def is_leaf(self):
    return self.value is not None


class XGBoostTree:

  def __init__(self, max_depth=3, reg_lambda=1.0, gamma=0.0):
    self.max_depth = max_depth
    self.reg_lambda = reg_lambda
    self.gamma = gamma
    self.root = None

  def _calc_leaf_weight(self, g, h):
    # w_j* = - sum(g_i) / (sum(h_i) + lambda)
    return -np.sum(g) / (np.sum(h) + self.reg_lambda)

  def _calc_split_gain(self, G_L, H_L, G_R, H_R):
    # Gain = 0.5 * [ G_L^2/(H_L + lambda) + G_R^2/(H_R + lambda) - (G_L+G_R)^2/(H_L+H_R+lambda) ] - gamma
    lam = self.reg_lambda
    score_L = (G_L**2) / (H_L + lam)
    score_R = (G_R**2) / (H_R + lam)
    score_parent = ((G_L + G_R) ** 2) / (H_L + H_R + lam)

    return 0.5 * (score_L + score_R - score_parent) - self.gamma

  def _best_split(self, X, g, h):
    best_gain = 0.0
    best_feat, best_thresh = None, None

    n_samples, n_features = X.shape

    for feat_idx in range(n_features):
      thresholds = np.unique(X[:, feat_idx])

      for thresh in thresholds:
        left_mask = X[:, feat_idx] <= thresh
        right_mask = ~left_mask

        if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
          continue

        G_L, H_L = np.sum(g[left_mask]), np.sum(h[left_mask])
        G_R, H_R = np.sum(g[right_mask]), np.sum(h[right_mask])

        gain = self._calc_split_gain(G_L, H_L, G_R, H_R)

        if gain > best_gain:
          best_gain = gain
          best_feat = feat_idx
          best_thresh = thresh

    return best_feat, best_thresh, best_gain

  def _build_tree(self, X, g, h, depth=0):
    if depth >= self.max_depth or len(g) <= 1:
      return XGBoostNode(value=self._calc_leaf_weight(g, h))

    feat_idx, thresh, gain = self._best_split(X, g, h)

    # Prune if gain is not strictly positive
    if feat_idx is None or gain <= 0:
      return XGBoostNode(value=self._calc_leaf_weight(g, h))

    left_mask = X[:, feat_idx] <= thresh
    left_child = self._build_tree(
        X[left_mask], g[left_mask], h[left_mask], depth + 1
    )
    right_child = self._build_tree(
        X[~left_mask], g[~left_mask], h[~left_mask], depth + 1
    )

    return XGBoostNode(
        feature=feat_idx,
        threshold=thresh,
        left=left_child,
        right=right_child,
    )

  def fit(self, X, g, h):
    self.root = self._build_tree(X, g, h)

  def _predict_row(self, x, node):
    if node.is_leaf():
      return node.value
    if x[node.feature] <= node.threshold:
      return self._predict_row(x, node.left)
    return self._predict_row(x, node.right)

  def predict(self, X):
    return np.array([self._predict_row(x, self.root) for x in X])


class XGBoostRegressorScratch:

  def __init__(
      self,
      n_estimators=50,
      learning_rate=0.1,
      max_depth=3,
      reg_lambda=1.0,
      gamma=0.0,
  ):
    self.n_estimators = n_estimators
    self.learning_rate = learning_rate
    self.max_depth = max_depth
    self.reg_lambda = reg_lambda
    self.gamma = gamma
    self.base_pred = 0.0
    self.trees = []

  def fit(self, X, y):
    # Initialize base prediction (e.g., mean of y or 0.0)
    self.base_pred = np.mean(y)
    y_hat = np.full(y.shape, self.base_pred)
    self.trees = []

    for _ in range(self.n_estimators):
      # For Mean Squared Error: L = 0.5 * (y - y_hat)^2
      # First-order gradient:  g_i = dL / dy_hat = -(y - y_hat) = y_hat - y
      g = y_hat - y

      # Second-order Hessian:  h_i = d^2L / dy_hat^2 = 1.0 (constant for MSE)
      h = np.ones_like(y)

      # Fit 2nd-order Taylor regularized tree
      tree = XGBoostTree(
          max_depth=self.max_depth,
          reg_lambda=self.reg_lambda,
          gamma=self.gamma,
      )
      tree.fit(X, g, h)

      # Update predictions with shrinkage
      y_hat += self.learning_rate * tree.predict(X)
      self.trees.append(tree)

  def predict(self, X):
    preds = np.full(X.shape[0], self.base_pred)
    for tree in self.trees:
      preds += self.learning_rate * tree.predict(X)
    return preds


# =========================================================================
# DEMO EXECUTION
# =========================================================================
if __name__ == "__main__":
  np.random.seed(42)

  # Non-linear synthetic regression dataset: y = sin(x) + noise
  X_train = np.linspace(-3, 3, 100).reshape(-1, 1)
  y_train = np.sin(X_train).ravel() + np.random.normal(0, 0.1, X_train.shape[0])

  # Train 2nd-order regularized XGBoost model from scratch
  xgb = XGBoostRegressorScratch(
      n_estimators=40,
      learning_rate=0.1,
      max_depth=3,
      reg_lambda=1.0,  # L2 Leaf Regularization
      gamma=0.01,  # Minimum Split Gain Penalty
  )
  xgb.fit(X_train, y_train)

  # Generate predictions
  y_pred = xgb.predict(X_train)
  mse = np.mean((y_train - y_pred) ** 2)

  print("=" * 65)
  print("  XGBOOST REGRESSOR FROM SCRATCH (2ND-ORDER TAYLOR ENGINE)")
  print("=" * 65)
  print(f"Number of Boosting Stages (Trees) : {xgb.n_estimators}")
  print(f"Learning Rate (eta)               : {xgb.learning_rate}")
  print(f"L2 Regularization (lambda)        : {xgb.reg_lambda}")
  print(f"Leaf Split Penalty (gamma)        : {xgb.gamma}")
  print(f"Final Mean Squared Error (MSE)    : {mse:.6f}")
  print("=" * 65)

  XGBOOST REGRESSOR FROM SCRATCH (2ND-ORDER TAYLOR ENGINE)
Number of Boosting Stages (Trees) : 40
Learning Rate (eta)               : 0.1
L2 Regularization (lambda)        : 1.0
Leaf Split Penalty (gamma)        : 0.01
Final Mean Squared Error (MSE)    : 0.005050
